# Day 92 — Image Representation

This notebook covers Digital Image Representation, Pixels & Channels, Channel-Last (`HWC`/`BHWC`) vs. Channel-First (`CHW`/`NCHW`) Tensor Layouts, Axis Permutation (`.permute()`), Pixel Rescaling (`[0, 255]` to `[0.0, 1.0]`), and Per-Channel Standardization (`(x - mean) / std`)

## Validation goals
- Contrast channel-last (`HWC`/`BHWC`) NumPy/PIL conventions against channel-first (`CHW`/`NCHW`) PyTorch tensor expectations for 2D spatial operations
- Implement `uint8_rgb_to_chw_float` to convert raw `uint8` image arrays into `torch.float32` tensors scaled to `[0.0, 1.0]` with contiguous `(C, H, W)` memory layout
- Implement `normalize_chw` using broadcastable `(C, 1, 1)` mean and standard deviation tensors to standardize RGB channels cleanly
- Verify tensor shape and value transformations across grayscale `(1, 28, 28)` and batched RGB `(16, 3, 64, 64)` tensors

### 1. Pixels, Channels, and Spatial Grids

A digital image is a grid of picture elements (`pixels`) arranged across height ($H$) and width ($W$):
- Grayscale Image: Each pixel holds a single intensity value (for example, 0 for black to 255 for white in an 8-bit integer image), giving a 2D spatial grid of shape (`H, W`) or a single-channel image with channel count C = 1.
- RGB Color Image: Each pixel stores three intensity values—Red, Green, and Blue—forming C = 3 color channels across a spatial grid of height $H$ and width $W$.

### 2. Channel-Last (NumPy/PIL) vs. Channel-First (PyTorch)

A frequent source of bugs in computer vision is mixing up axis order between image libraries and PyTorch layers:
- Image / NumPy / PIL Convention (`HWC or BHWC`): Channels sit at the last axis—(Height, Width, Channels) for a single image, or (Batch, Height, Width, Channels) for a batch.
- PyTorch Convention (`CHW or NCHW`): PyTorch's nn.Conv2d expects channels before spatial dimensions—(Channels, Height, Width) for one image, or (Batch, Channels, Height, Width) for a mini-batch.

Use .permute(...) to reorder axes explicitly when converting between HWC and CHW.

### 3. Pixel Scaling and Normalization

Raw images often start as uint8 values in [0, 255]. Passing large unscaled integers directly into a network makes gradients and activations harder to keep well-behaved:Scale to [0.0, 1.0] (float32): 
- Divide pixel intensities by 255.0 and cast to torch.float32.
- Channel Normalization: Shift and scale each channel using mean $\mu$ and standard deviation $\sigma$:$$x_{\text{norm}} = \frac{x - \mu}{\sigma}$$

Standardizing inputs keeps activations near zero with unit scale, helping gradient-based optimization converge faster and more reliably.

### 4. Implementing Image Representation & Tensor Layout

Let's write a self-contained script that builds synthetic grayscale and RGB image arrays, converts between HWC and CHW layouts, normalizes pixel values, and checks batch tensor shapes.

In [1]:
import numpy as np
import torch


def uint8_rgb_to_chw_float(img_hwc: np.ndarray) -> torch.Tensor:
    """Converts a uint8 (H, W, C) image array into a float32 (C, H, W) tensor in [0, 1]."""
    if img_hwc.ndim != 3:
        raise ValueError(f"Expected 3D (H, W, C) array, got shape {img_hwc.shape}")

    tensor_hwc = torch.from_numpy(img_hwc).to(dtype=torch.float32) / 255.0
    return tensor_hwc.permute(2, 0, 1).contiguous()


def normalize_chw(tensor_chw: torch.Tensor, mean: list[float], std: list[float]) -> torch.Tensor:
    """Applies per-channel standardization: (x - mean) / std on a (C, H, W) tensor."""
    if tensor_chw.ndim != 3:
        raise ValueError(f"Expected 3D (C, H, W) tensor, got shape {tensor_chw.shape}")
    if len(mean) != tensor_chw.shape[0] or len(std) != tensor_chw.shape[0]:
        raise ValueError("mean and std lengths must match channel count C.")

    mean_t = torch.tensor(mean, dtype=tensor_chw.dtype, device=tensor_chw.device).view(-1, 1, 1)
    std_t = torch.tensor(std, dtype=tensor_chw.dtype, device=tensor_chw.device).view(-1, 1, 1)
    return (tensor_chw - mean_t) / std_t


if __name__ == "__main__":
    rng = np.random.default_rng(42)

    # Single grayscale image: H=28, W=28 -> add channel dim for (1, 28, 28)
    gray_hw = rng.integers(0, 256, size=(28, 28), dtype=np.uint8)
    gray_chw = torch.from_numpy(gray_hw).to(dtype=torch.float32).unsqueeze(0) / 255.0

    # Batch of 16 RGB images: (16, 64, 64, 3) in HWC uint8 format
    batch_bhwc = rng.integers(0, 256, size=(16, 64, 64, 3), dtype=np.uint8)
    batch_bchw = torch.from_numpy(batch_bhwc).to(dtype=torch.float32).permute(0, 3, 1, 2) / 255.0

    rgb_mean = [0.485, 0.456, 0.406]
    rgb_std = [0.229, 0.224, 0.225]
    batch_norm = normalize_chw(batch_bchw[0], rgb_mean, rgb_std).unsqueeze(0)

    print("--- Day 92: Image Representation & Tensor Shapes ---")
    print(f"Grayscale CHW shape: {tuple(gray_chw.shape)}, dtype={gray_chw.dtype}, range=[{gray_chw.min():.2f}, {gray_chw.max():.2f}]")
    print(f"Batch BHWC shape:    {tuple(batch_bhwc.shape)}, dtype={batch_bhwc.dtype}")
    print(f"Batch BCHW shape:    {tuple(batch_bchw.shape)}, dtype={batch_bchw.dtype}")
    print(f"Single norm NCHW:    {tuple(batch_norm.shape)}, min={batch_norm.min():.2f}, max={batch_norm.max():.2f}")

--- Day 92: Image Representation & Tensor Shapes ---
Grayscale CHW shape: (1, 28, 28), dtype=torch.float32, range=[0.00, 1.00]
Batch BHWC shape:    (16, 64, 64, 3), dtype=uint8
Batch BCHW shape:    (16, 3, 64, 64), dtype=torch.float32
Single norm NCHW:    (1, 3, 64, 64), min=-2.12, max=2.64


### Guided Question:

Why does PyTorch expect (`Batch, Channels, Height, Width`) (`NCHW`) for image batches when many image loaders or NumPy arrays use (`Height, Width, Channels`) (`HWC`)?

### Answer:

PyTorch defaults to (Batch, Channels, Height, Width) (`NCHW`) largely for historical GPU performance reasons and cuDNN alignment, whereas image files and NumPy arrays naturally put channels last as (Height, Width, Channels) (`HWC`) for human ergonomics and spatial indexing.

**Why Image Loaders and NumPy Often Use HWC**

- Spatial-first mental model and indexing: Images are 2D spatial grids (y row, x column) where each pixel holds a small tuple of values (such as (R, G, B)). Indexing img[y, x] to get a pixel vector feels natural and matches how raster scanlines are stored in files.
- Ecosystem conventions: Many image decoding and display libraries—including Pillow/PIL, OpenCV, and Matplotlib—represent standard images with height and width first and channels last (often (`H, W, C`) or (`H, W`) for grayscale).

**Why PyTorch Defaults to NCHW**

- Legacy cuDNN and GPU memory layout history: Early GPU deep learning libraries—most notably NVIDIA's cuDNN—were heavily optimized around `NCHW` for 2D convolutions and spatial pooling operations. Making `NCHW` the default in PyTorch avoided constant layout transpositions before calling backend convolution kernels.
- Contiguous per-channel planes: In `NCHW`, all spatial values for a single channel sit in a contiguous 2D slice (`H x W`), which historically aligned well with vectorized `SIMD/SIMT` execution and spatial kernel traversal patterns across channels.

**Modern Nuance: channels_last (NHWC)**

NCHW is still PyTorch's default tensor dimension order, so layers like nn.Conv2d expect (`N, C, H, W`) by default. However, on modern GPUs with Tensor Cores (such as Volta, Ampere, Hopper, and newer architectures), PyTorch also supports a channels_last memory format (NHWC logical layout with strided memory layout) via `tensor.to(memory_format=torch.channels_last)` and `model.to(memory_format=torch.channels_last)`. That keeps the familiar (`N, C, H, W`) shape API while laying out underlying memory in a way that can speed up Tensor Core convolutions.

**Practical Takeaway When Converting**

When moving data from a NumPy image batch (N, H, W, C) or a single (H, W, C) array into a standard PyTorch Conv2d pipeline, explicitly permute axes—for example np.transpose(batch, (`0, 3, 1, 2`)) or tensor.permute(`0, 3, 1, 2`)—so channels match (`N, C, H, W`).

### Guided Question:

What happens to numerical values when we first divide uint8 pixels (`0–255`) by `255.0 `and then apply `(x - mean) / std` per channel?

### Answer:

Dividing uint8 pixels (`0 to 255`) by `255.0` first converts the values from integers in `[0, 255]` into floating-point numbers in `[0.0, 1.0]`. Subtracting mean and dividing by std per channel then shifts and rescales each channel so its values are roughly centered near 0 with spread on the order of 1.

**Step-by-Step Numerical Effect**

Rescaling to `[0.0, 1.0]` (`x_norm = x / 255.0`):
- The integer type (uint8) cannot hold fractions or negative numbers, and arithmetic on uint8 wraps modulo 256 instead of overflowing cleanly. Dividing by 255.0 promotes the data to a floating-point type (such as float32) and maps `0 -> 0.0, 255 -> 1.0`, and intermediate intensities proportionally across `[0.0, 1.0]`.
- Because mean and std in image pipelines (such as ImageNet statistics like mean = `[0.485, 0.456, 0.406]` and std = `[0.229, 0.224, 0.225]`) are defined on the `[0.0, 1.0]` scale, scaling by 255.0 first puts the pixels into the expected range.

**Per-channel standardization (x_std = (x_norm - mean_c) / std_c):**
- For each channel c, subtracting mean_c shifts that channel's values so pixels equal to mean_c become 0.0. Pixels darker than mean_c become negative, and pixels brighter than mean_c become positive.
- Dividing by std_c rescales the spread of channel c. If a channel has a smaller std_c (say 0.20), a 0.10 shift from the mean becomes 0.5 after division; if std_c is larger, the resulting magnitude is smaller.
- Because mean_c and std_c differ across channels (for instance, red, green, and blue), each channel is shifted and scaled by its own offset and spread rather than a single global constant.

**Numerical Example on a Single Channel**

Suppose a red-channel pixel starts as uint8 value 128, with mean_R = 0.5 and std_R = 0.25:
- Step 1 (Divide by 255.0): 128 / 255.0 ≈ 0.502 (now in [0.0, 1.0]).
- Step 2 (Subtract mean_R): 0.502 - 0.5 = +0.002 (centered around 0).
- Step 3 (Divide by std_R): 0.002 / 0.25 = +0.008 (rescaled by channel spread).

**For extreme pixels on that same channel:**

- A 0 pixel becomes `(0.0 - 0.5) / 0.25 = -2.0`.
- A 255 pixel becomes `(1.0 - 0.5) / 0.25 = +2.0`.

**Why This Matters During Training**

- Balanced feature scales across channels and layers: Channels with naturally higher brightness or wider contrast no longer dominate dot products or early layer activations solely due to scale.
- Well-conditioned activations and gradients: Keeping inputs near zero with moderate variance helps prevent saturated activations, large initial weight updates, and unstable gradients in early training steps.
- Consistent statistics across splits: Computing or applying mean and std consistently across training, validation, and test data (using training-set statistics rather than recomputing per image or per test batch) keeps the numerical mapping fixed at inference time.

### Guided Question:

If a single uint8 RGB image has shape `(32, 32, 3)`, what is its shape after converting to CHW and stacking 8 images into a training batch?

### Answer:

The resulting batch shape is `(8, 3, 32, 32)` (`(Batch, Channels, Height, Width)`).

**How the Shape Changes**

- Single image `(32, 32, 3)` (HWC): Height is 32, width is 32, and there are 3 color channels `(R, G, B)`.
- Permute to CHW (`(3, 32, 32)`): Moving the channel axis to the front places the 3 channels first, followed by height 32 and width 32.
- Stack 8 images (`(8, 3, 32, 32)`): Adding a leading batch dimension of size 8 along axis 0 (for example via torch.stack or np.stack) produces `(8, 3, 32, 32)`.

**Dtype Note**

Stacking or permuting by itself preserves the data type (`uint8` unless converted to `float32` via `.float()` or division by `255.0`), whereas the shape becomes `(8, 3, 32, 32)`.